In [13]:
# =========================================================
# Fine-tune Whisper-Medium with LoRA (No QLoRA)
# =========================================================

# !pip install -q peft accelerate evaluate jiwer

import os
import gc
import torch
import torchaudio
import pandas as pd
import evaluate
from dataclasses import dataclass
from typing import Any
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

# Clear memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================================================
# CONFIGURATION (Local PC Paths)
# =========================================================
PROJECT_ROOT = r"d:\Buet_Datathon"
MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
CHUNK_CSV = os.path.join(PROJECT_ROOT, "data", "processed", "train_final.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models", "whisper-bengali-lora")
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if device == "cuda" else torch.float32
MAX_LABEL_LEN = 448

# =========================================================
# 1. LOAD DATASET
# =========================================================
print("Loading dataset...")
if not os.path.exists(CHUNK_CSV):
    raise FileNotFoundError(f"Missing CSV: {CHUNK_CSV}")

df = pd.read_csv(CHUNK_CSV)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_limit = int(len(df) * 0.9)
train_df = df.iloc[:train_limit]
eval_df = df.iloc[train_limit:]

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

# =========================================================
# 2. LOAD BASE MODEL (LoRA: no quantization)
# =========================================================
print("Loading base model...")
processor = WhisperProcessor.from_pretrained(MODEL_ID)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
).to(device)
model.config.use_cache = False
model.to(device)

# =========================================================
# 3. APPLY LORA
# =========================================================
print("Applying LoRA...")
lora_config = LoraConfig(
    r=32,                      # you can try 32 later
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =========================================================
# 4. PREPROCESSING
# =========================================================
def prepare_dataset(batch):
    try:
        audio_path = batch["filename"]
        wav, sr = torchaudio.load(audio_path)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)

        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)

        batch["input_features"] = processor.feature_extractor(
            wav.numpy(),
            sampling_rate=16000
        ).input_features[0]

        batch["labels"] = processor.tokenizer(
            batch["transcript"],
            truncation=True,
            max_length=MAX_LABEL_LEN
        ).input_ids

    except Exception as e:
        print(f"Error processing {batch.get('filename', 'unknown')}: {e}")
        batch["input_features"] = None
        batch["labels"] = None

    return batch

print("Processing data...")
train_dataset = train_dataset.map(prepare_dataset)
eval_dataset = eval_dataset.map(prepare_dataset)

# Important fix: filter valid rows only
train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)
eval_dataset = eval_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)

print(f"Train usable: {len(train_dataset)}")
print(f"Eval usable : {len(eval_dataset)}")

# =========================================================
# 5. DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    feature_dtype: torch.dtype = torch.float16

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # force same dtype as model
        batch["input_features"] = batch["input_features"].to(self.feature_dtype)

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    feature_dtype=MODEL_DTYPE
)

# =========================================================
# 6. METRICS
# =========================================================
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# =========================================================
# 7. TRAINING ARGUMENTS (LoRA-friendly)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=3000,
    gradient_checkpointing=True,
    fp16=(device == "cuda"),
    fp16_full_eval=(device == "cuda"),
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

# =========================================================
# 8. START TRAINING
# =========================================================
print("Starting LoRA Training...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Training Complete! Model saved to {OUTPUT_DIR}")


Loading dataset...
Train samples: 11250
Eval samples: 1250
Loading base model...


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 967.53it/s, Materializing param=model.encoder.layers.23.self_attn_layer_norm.weight]    


Applying LoRA...
trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204
Processing data...


Filter: 100%|██████████| 1250/1250 [01:10<00:00, 17.62 examples/s]


Train usable: 11250
Eval usable : 1250
Starting LoRA Training...


Step,Training Loss,Validation Loss,Wer
500,1.479201,0.382959,36.207617
1000,1.379077,0.357865,34.753334
1500,1.290115,0.348728,32.784830
2000,1.241599,0.342403,32.984091
2500,1.212829,0.341019,33.403503
3000,1.164485,0.340172,32.487546


Training Complete! Model saved to d:\Buet_Datathon\models\whisper-bengali-lora


In [ ]:
import os
import glob
import re
import gc
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# -------------------------
# CONFIGURATION (PC PATHS)
# -------------------------
PROJECT_ROOT = r"d:\Buet_Datathon"
TEST_AUDIO_DIR = os.path.join(
    PROJECT_ROOT, "data", "raw", "bengali_asr", "transcription", "transcription", "test", "audio"
)
ADAPTER_DIR = os.path.join(PROJECT_ROOT, "models", "whisper-bengali-lora")  # LoRA adapter dir
BASE_MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
OUT_CSV = os.path.join(PROJECT_ROOT, "data", "processed", "submission242.csv")
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

BATCH_SIZE = 16
CHUNK_SEC = 20.0
OVERLAP_SEC = 1.0
SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device} | Batch Size: {BATCH_SIZE}")

# -------------------------
# 1. LOAD MODELS (LoRA)
# -------------------------
print("Loading Whisper + LoRA...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID)

base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
).to(device)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="bengali", task="transcribe")
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.suppress_tokens = []
model.generation_config.begin_suppress_tokens = []
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None

print("Loading VAD...")
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    onnx=False,
)
(get_speech_timestamps, _, _, _, collect_chunks) = utils
vad_model = vad_model.to(device).eval()

# -------------------------
# 2. PREPARE CHUNK LIST
# -------------------------
print("Scanning files to create chunk list...")
test_files = sorted(glob.glob(os.path.join(TEST_AUDIO_DIR, "*.wav")))
all_chunks = []

for wav_path in tqdm(test_files, desc="Preparing Metadata"):
    file_id = os.path.splitext(os.path.basename(wav_path))[0]
    info = torchaudio.info(wav_path)
    total_samples = info.num_frames
    orig_sr = info.sample_rate

    target_samples = int(total_samples * (SR / orig_sr))
    chunk_samples = int(CHUNK_SEC * SR)
    stride_samples = int((CHUNK_SEC - OVERLAP_SEC) * SR)

    for start in range(0, target_samples, stride_samples):
        end = min(start + chunk_samples, target_samples)
        if (end - start) < int(1.0 * SR):
            continue
        all_chunks.append({"file_id": file_id, "path": wav_path, "start": start, "end": end})

print(f"Total Chunks to process: {len(all_chunks)}")

# -------------------------
# 3. HELPERS
# -------------------------
def load_and_crop(path, start, end):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.squeeze(0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    return wav[start:end]

def apply_vad_and_merge(wav_tensor):
    if wav_tensor.numel() == 0:
        return None
    with torch.inference_mode():
        wav_gpu = wav_tensor.to(device)
        timestamps = get_speech_timestamps(wav_gpu, vad_model, sampling_rate=SR, threshold=0.5)
        if not timestamps:
            return None
        merged = collect_chunks(timestamps, wav_gpu)
        return merged.cpu()

# -------------------------
# 4. BATCH INFERENCE LOOP
# -------------------------
results_map = {fid: [] for fid in set(c["file_id"] for c in all_chunks)}

for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Batch Inference"):
    batch_meta = all_chunks[i:i + BATCH_SIZE]
    valid_features = []
    valid_indices = []

    for idx, meta in enumerate(batch_meta):
        raw_wav = load_and_crop(meta["path"], meta["start"], meta["end"])
        clean_wav = apply_vad_and_merge(raw_wav)

        if clean_wav is not None and clean_wav.numel() > 0:
            feat = processor(
                clean_wav.numpy(),
                sampling_rate=SR,
                return_tensors="pt",
            ).input_features[0]
            valid_features.append(feat)
            valid_indices.append(idx)

    if not valid_features:
        continue

    input_tensor = torch.stack(valid_features).to(device, dtype=MODEL_DTYPE)

    with torch.inference_mode():
        if device == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                generated_ids = model.generate(
                    input_tensor,
                    max_new_tokens=128,
                    num_beams=5,
                    do_sample=False,
                )
        else:
            generated_ids = model.generate(
                input_tensor,
                max_new_tokens=128,
                num_beams=5,
                do_sample=False,
            )

    transcripts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for j, text in enumerate(transcripts):
        meta_idx = valid_indices[j]
        file_id = batch_meta[meta_idx]["file_id"]
        results_map[file_id].append(text.strip())

    del input_tensor, generated_ids, valid_features
    if i % 10 == 0 and device == "cuda":
        torch.cuda.empty_cache()

# -------------------------
# 5. ASSEMBLE SUBMISSION
# -------------------------
final_rows = []
for file_id in sorted(results_map.keys()):
    full_text = " ".join(results_map[file_id])
    full_text = re.sub(r"\s+", " ", full_text).strip()
    final_rows.append({"filename": file_id, "transcript": full_text})

df_sub = pd.DataFrame(final_rows)
df_sub.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(f"Submission saved to: {OUT_CSV}")
print(df_sub.head())


Device: cuda | Batch Size: 24
Loading Whisper + LoRA...


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 1102.33it/s, Materializing param=model.encoder.layers.23.self_attn_layer_norm.weight]   


Loading VAD...


Using cache found in C:\Users\T2430518/.cache\torch\hub\snakers4_silero-vad_master


Scanning files to create chunk list...


Preparing Metadata: 100%|██████████| 24/24 [00:00<00:00, 7711.30it/s]


Total Chunks to process: 4214


Batch Inference: 100%|██████████| 176/176 [4:58:37<00:00, 101.80s/it]

Submission saved to: d:\Buet_Datathon\data\processed\submission242.csv
   filename                                         transcript
0  test_001  এক্সকিউরি এটা নিতে পারে এটা আপনাকে বেস ভালো মা...
1  test_002  আমি আমি কিন্তু ইচ্ছাধারী নাগিন তাই নাগ আমি কিন...
2  test_003  গল্পটির স্বত্য আনন্দ পাবলিশার্স প্রাইভেট লিমিট...
3  test_004  যেতে রাতের ট্রেনই আমাদের সবচাইতে পছন্দের বেশ খ...
4  test_005  নিবেদন প্রাইডে ক্লাসিক্স ওরে বাপ রে বাপ কি মেঘ...


In [21]:
import re
import unicodedata
import pandas as pd
from difflib import SequenceMatcher

INPUT_CSV = r"d:\Buet_Datathon\data\processed\submission2425.csv"
OUTPUT_CSV = r"d:\Buet_Datathon\data\processed\submission_clean_fina2425.csv"

def norm_key(x: str) -> str:
    x = unicodedata.normalize("NFC", str(x)).lower()
    x = re.sub(r"[^\u0980-\u09FFa-z0-9]", "", x)
    return x

FILLERS = {"ভাই","না","হ্যাঁ","এই","ওই","আরে","মানে","ওকে","প্লিজ","চলো","দাঁড়া","দাঁড়া","হুম"}
FILLER_KEYS = {norm_key(x) for x in FILLERS}

PHRASE_FIXES = [
    (r"এক্সকিউরি", "এক্সকিউজ মি"),
    (r"ড্রেস টুস", "ড্রেস"),
    (r"সেলস ম্যান", "সেলসম্যান"),
    (r"গুড মর্ডিং", "গুড মর্নিং"),
    (r"হেরেজমেন্ট|হেরেজমেন|হেরেজমেনট", "হ্যারাসমেন্ট"),
]

WORD_FIXES = {
    "রেসিভ": "রিসিভ",
    "পাসে": "পাশে",
    "সেন্টার": "সেন্টার",
    "বিজনেসস": "বিজনেস",
    "স্যরি": "সরি",
}

def normalize_text(t: str) -> str:
    t = unicodedata.normalize("NFC", str(t or ""))
    t = t.replace("\uFFFD", "")        # remove �
    t = re.sub(r"<[^>]*>", " ", t)     # remove <> blocks
    t = t.replace("<>", " ")
    t = re.sub(r"[\x00-\x1F\x7F]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def apply_maps(text: str) -> str:
    for p, r in PHRASE_FIXES:
        text = re.sub(p, r, text)
    return text

# FILLER words কখনো remove/cap করা হবে না
def cap_runs(tokens, short_max=1, normal_max=2):
    out = []
    prev, run = None, 0

    for tok in tokens:
        k = norm_key(tok)
        if not k:
            continue

        if k in FILLER_KEYS:
            out.append(tok)
            prev, run = None, 0
            continue

        if k == prev:
            run += 1
        else:
            prev, run = k, 1

        mx = short_max if len(k) <= 2 else normal_max
        if run <= mx:
            out.append(tok)
    return out

# যেসব ngram-এ filler আছে সেগুলো untouched
def remove_ngram_loops(tokens, min_n=2, max_n=12, min_repeats=2):
    norms = [norm_key(t) for t in tokens]
    out = []
    i, L = 0, len(tokens)

    while i < L:
        reduced = False
        for n in range(min(max_n, L - i), min_n - 1, -1):
            if i + 2 * n > L:
                continue
            base = norms[i:i+n]
            if not any(base):
                continue
            if any(x in FILLER_KEYS for x in base):
                continue

            reps = 1
            j = i + n
            while j + n <= L and norms[j:j+n] == base:
                reps += 1
                j += n

            if reps >= min_repeats:
                out.extend(tokens[i:i+n])  # keep one copy
                i = j
                reduced = True
                break
        if not reduced:
            out.append(tokens[i])
            i += 1
    return out

# overlap echo remove করবে, তবে filler-involved window skip করবে
def remove_overlap_echo(tokens, min_w=4, max_w=16, sim_thr=0.94):
    norms = [norm_key(t) for t in tokens]
    out_t, out_n = [], []
    i, L = 0, len(tokens)

    while i < L:
        skipped = False
        wmax = min(max_w, len(out_n), L - i)

        for w in range(wmax, min_w - 1, -1):
            tail = out_n[-w:]
            head = norms[i:i+w]

            if any(x in FILLER_KEYS for x in tail) or any(x in FILLER_KEYS for x in head):
                continue

            if tail == head:
                i += w
                skipped = True
                break

            if SequenceMatcher(None, " ".join(tail), " ".join(head)).ratio() >= sim_thr:
                i += w
                skipped = True
                break

        if not skipped:
            out_t.append(tokens[i])
            out_n.append(norms[i])
            i += 1

    return out_t

def postprocess(t: str) -> str:
    t = normalize_text(t)
    t = apply_maps(t)

    tokens = t.split()

    mapped = []
    for tok in tokens:
        k = norm_key(tok)
        mapped.append(WORD_FIXES.get(k, tok))
    tokens = mapped

    norms = [norm_key(x) for x in tokens if norm_key(x)]
    uniq_ratio = len(set(norms)) / max(len(norms), 1)
    aggressive = uniq_ratio < 0.45

    if aggressive:
        tokens = cap_runs(tokens, short_max=1, normal_max=1)
        tokens = remove_ngram_loops(tokens, max_n=14, min_repeats=2)
        tokens = remove_overlap_echo(tokens, max_w=20, sim_thr=0.90)
    else:
        tokens = cap_runs(tokens, short_max=1, normal_max=2)
        tokens = remove_ngram_loops(tokens, max_n=10, min_repeats=2)
        tokens = remove_overlap_echo(tokens, max_w=14, sim_thr=0.96)

    tokens = cap_runs(tokens, short_max=1, normal_max=2)
    return normalize_text(" ".join(tokens))

df = pd.read_csv(INPUT_CSV)
df["transcript"] = df["transcript"].fillna("").map(postprocess)

final_df = df[["filename", "transcript"]].copy()
final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_CSV)
print("Remaining �:", final_df["transcript"].str.count("\uFFFD").sum())
print(final_df.head())


Saved: d:\Buet_Datathon\data\processed\submission_clean_fina2425.csv
Remaining �: 0
   filename                                         transcript
0  test_001  এটা আপনাকে বেশ ভালো মানাবে এমনি থেকে ভীষণ সুন্...
1  test_002  আমি কিন্তু ইচ্ছাকারী না কিন্তু ছোল দিলে তোমরা ...
2  test_003  গল্পটির স্বত্ত্ব আনন্দ পাবলিশার্স প্রাইভেট লিম...
3  test_004  যে কোন জায়গায় যেতে রাতের ট্রেনই আমাদের সবচাই...
4  test_005  নিবেদন প্রাইডে ক্লাসিক্স ওরে বাপ রে বা কি মেঘ ...
